# Day 1 · Module 3: Subagents (Explorer / Critic, Coupling Discovery)

**Objective:** define two read-only subagents by their tool whitelist alone, use one to map a codebase surface and the other to attack a plan against it, and see whether either one surfaces a real cross-module coupling.


## Relevant Claude Code Docs
Review these before starting the module:
- [Create custom subagents](https://code.claude.com/docs/en/sub-agents)
- [Run agents in parallel](https://code.claude.com/docs/en/agents)
- [Orchestrate subagents at scale with dynamic workflows](https://code.claude.com/docs/en/workflows)

## 1 · The idea

A subagent is not a personality — it is a tool whitelist plus a role description. Giving an "explorer" only `Read, Grep, Glob` is what makes it safe to point at untrusted or unfamiliar code: it can map a surface without ever being able to change it. The same is true of a "critic" — its value comes entirely from having no write access and no incentive to be agreeable.

Isolating each subagent's context also matters — and the set of files it reads is the real variable behind "exploration quality," not how eloquent its report sounds. An explorer pointed only at `analytics.py` will come back with a confident, wrong map, because the fact that matters here lives in `reporting.py`, not in the file it was told to look at. A critic that reviews a plan without having read the exploration will repeat the same blind spot instead of catching it.


### Tools-as-role, verbatim

This isn't prose — it's the actual file. Run the cell below to print the real frontmatter from `starter-agents/explorer.skeleton.md` and `starter-agents/critic.skeleton.md`. Both declare `tools: Read, Grep, Glob` and nothing else.


In [ ]:
import pathlib
agents = pathlib.Path(proj) / "starter-agents"
for name in ("explorer.skeleton.md", "critic.skeleton.md"):
    body = (agents / name).read_text()
    frontmatter = body.split("---")[1].strip()
    print(f"--- {name} frontmatter ---")
    print(frontmatter)
    print()


The `tools:` line above **is** the safety boundary — not a comment near it, not a note in the system prompt below it. Adding `Write` to that line is exactly the change that would break the trust: the moment an "explorer" can write, its report is no longer a disinterested map, it's an artifact from something that could have edited the evidence. When you complete the skeletons in the exercise below, the only thing you're allowed to fill in is the `# TODO` report/finding schema in the body — never the `tools:` line itself.


### Grounding

The ticket for this module asks you to map what a change to analytics region bucketing would touch, before anyone writes code. The plan under review (`reference/m3/plan.md`) assumes the bucketing logic is local to `reporting.py` already — and it is, but the plan also assumes no other module needs to change. It's wrong about that: `analytics.summary()` imports `bucket_country` from `reporting.py`, and `reporting.monthly_statement()` calls the same function. There is exactly one bucket table, owned by `reporting`, and both the per-account activity summary and the regulatory monthly statement read through it — so a change made "inside analytics" would actually need to land in `reporting`, and would silently affect the monthly statement too.

Separately, `posting.post()` authorizes a transfer against its hold, then records the ledger entry via `ledger.record_entry()` as its own, later commit — the two are not wrapped in one transaction, so a crash between authorization and the ledger write loses the entry without failing the post.

Run the cell below against the real source to confirm both facts before you start — never take a paragraph on faith when the source is one `Read` away.


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` right below, for the grounding check, and again later for the exercise and self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


With `proj` resolved, confirm the coupling and the non-atomic posting path directly against the real source, rather than taking the paragraph above on faith:


In [ ]:
import pathlib
src = pathlib.Path(proj) / "src" / "contoso"
an = (src / "analytics.py").read_text()
rep = (src / "reporting.py").read_text()
post = (src / "posting.py").read_text()

imports_bucket_country = "bucket_country" in an and "reporting" in an
print("analytics imports bucket_country:", imports_bucket_country)

statement_uses_it = "bucket_country" in rep and "monthly_statement" in rep
print("monthly_statement uses bucket_country:", statement_uses_it)

non_atomic = "ledger.record_entry" in post and "def post(" in post
print("posting.post records via ledger.record_entry, non-atomically:", non_atomic)


## 2 · Your exercise

Work through these steps in order:

1. Copy `reference/m3/plan.md` to `artifacts/m3/plan.md` — this is the plan under review; you did not write it and should not edit its content.
2. Copy `starter-agents/explorer.skeleton.md` to `.claude/agents/explorer.md` and `starter-agents/critic.skeleton.md` to `.claude/agents/critic.md`. Complete the `# TODO` report/finding schema in each body. Keep the frontmatter's `tools:` line exactly as printed above — do not add `Write` or `Edit`.
3. Run the explorer subagent on the scenario (`scenarios/m3-explore-analytics.md`) plus the plan (`artifacts/m3/plan.md`). Have it write its report to `artifacts/m3/exploration.md`.
4. Run the critic subagent on the plan plus the exploration report. Have it write its findings to `artifacts/m3/critique.md`, using the finding shape below.
5. Read the critique and mark each finding, inline, as legit, pedantic, or wrong — and say why in one line per finding.


### The critic's finding shape

Every finding the critic files should follow this template — fill it in for real findings, don't just describe the shape in prose:

```
severity: blocker | concern | note
finding: <one line, specific>
repro: <the concrete input, sequence, or scenario that exposes it>
ties to: <the plan line, assumption, or acceptance criterion it attacks>
```

A finding with a severity but no repro is an opinion. A finding with a repro but no tie to the plan is trivia — the critic's job is to attack the plan under review, not to free-associate about the codebase.


### What good looks like

A good exploration report names `reporting.py` and `bucket_country` explicitly as the hazard — not just "region logic is spread out" — and separately notes the non-atomic posting path through `ledger.record_entry`. A good critique does not praise; every finding carries a severity (blocker, concern, or note), a concrete reproduction, and a tie back to a line in the plan's stated intent or assumptions.

Common failure modes:
- An explorer that reads only `analytics.py` and never opens `reporting.py`, so it misses the shared function entirely and the coupling never surfaces.
- A subagent whose frontmatter or a later edit slips in `Write` or `Edit` tools — at that point it is no longer a safe read-only role, and its report can no longer be trusted as an unbiased map.
- A critic that summarizes what the plan gets right instead of only filing findings — approval is a different job, not this one.


### Fast-finisher

Re-run the explorer with a cheaper model instead of the default, on the same scenario and plan. Diff the two exploration reports, and explicitly record: **did the cheaper model open `reporting.py` at all?** Not just whether its prose mentions the coupling — check the tool calls or the file list in its report. This is a live, small-scale version of the token-economics point from the slides: cheaper tiers often save cost by reading less, and "exploration quality" quietly becomes "how many of the related files did the model bother to open."


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `.claude/agents/` and `artifacts/m3/`. Run it any time; it's safe to run before you've written anything. It also doubles as the read-only self-check: both subagents' frontmatter should show no `Write`/`Edit`.


In [ ]:
import pathlib
agents = pathlib.Path(proj) / ".claude" / "agents"
for name in ("explorer.md", "critic.md"):
    f = agents / name
    if f.exists():
        body = f.read_text()
        fm = body.split("---")[1] if body.count("---") >= 2 else ""
        ro = "tools:" in body and "Write" not in fm and "Edit" not in fm
        print(f"[x] {name} exists (read-only frontmatter: {'yes' if ro else 'CHECK THIS'})")
    else:
        print(f"[ ] {name} missing")
expl = pathlib.Path(proj) / "artifacts" / "m3" / "exploration.md"
if expl.exists():
    t = expl.read_text().lower()
    print(f"\nExploration present ({len(t.split())} words)")
    print(f"  [{'x' if 'reporting' in t or 'bucket_country' in t else ' '}] surfaced the analytics<-reporting coupling?")
    print(f"  [{'x' if 'posting' in t or 'ledger' in t or 'atomic' in t or 'commit' in t else ' '}] noticed the non-atomic posting path?")
else:
    print("\n[ ] artifacts/m3/exploration.md missing")


## 3 · Debrief

- What in the explorer's tool whitelist made its report trustworthy as a map, independent of how good its prose was?
- Did the critic's findings improve because it could see the exploration report, or would it have found the same things from the plan alone?
- Of the findings you marked "pedantic" or "wrong," what made them easy to dismiss — and could you have written a check that catches a critic doing that automatically?
- Did the cheaper-tier explorer open `reporting.py`? If not, what did that cost you, and was it worth what it saved?


---
### Take-home

A subagent's tools are its role definition, not a suggestion layered on top of one. Read-only access is what makes an explorer's map and a critic's findings trustworthy inputs to whatever comes next, instead of just more prose to double-check.

One more rule worth keeping from team orchestration, wherever you wire agents together: bound the disagreement. If a critic raises a blocker, the implementer gets exactly one revision loop; still blocked after that, escalate — never loop forever, never force green. The capstone's `run-pipeline.md` encodes exactly this.

(Related concept: Module 4 runs two implementations in parallel — a different use of isolated subagent context, this time for divergent solutions rather than divergent roles.)
